<a href="https://colab.research.google.com/github/agemagician/ProtTrans/blob/master/Embedding/PyTorch/Advanced/ProtT5-XL-UniRef50.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. 埋め込みベクトルの取得

## 1.1 必要なライブラリのインポート

In [ ]:
import torch
from transformers import T5EncoderModel, T5Tokenizer
import numpy as np
import os
from tqdm.auto import tqdm
from Bio import SeqIO

## 1.2 関数check_valid_seqの宣言

In [ ]:
exclusion_bases = list("BJOUXZ")

def check_valid_seq(seq):
    """
    有効なアミノ酸配列かどうかを返す関数
    :param seq: タンパク質配列
    :return: 有効なタンパク質ならば True
    """

    # 配列長が1000以上なら除外
    if len(seq) >= 1000:
        return False

    # "BJOUXZ" が含まれていたら除外
    for base in exclusion_bases:
        if base in seq:
            return False

    return True

## 1.3 モデルのロード

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

tokenizer = T5Tokenizer.from_pretrained("Rostlab/prot_t5_xl_uniref50", do_lower_case=False )
model = T5EncoderModel.from_pretrained("Rostlab/prot_t5_xl_uniref50")

if device == torch.device("cpu"):
    model.to(torch.float32)
    
model.eval()

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


T5EncoderModel(
  (shared): Embedding(128, 1024)
  (encoder): T5Stack(
    (embed_tokens): Embedding(128, 1024)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=1024, out_features=4096, bias=False)
              (k): Linear(in_features=1024, out_features=4096, bias=False)
              (v): Linear(in_features=1024, out_features=4096, bias=False)
              (o): Linear(in_features=4096, out_features=1024, bias=False)
              (relative_attention_bias): Embedding(32, 32)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=1024, out_features=16384, bias=False)
              (wo): Linear(in_features=16384, out_features=1024, bias=False)
              (dropout): Dropo

## 1.4 保存先ディレクトリの作成

In [ ]:
data_dir = "./gds_dataset_exclusion"  # データセットのディレクトリ
filenames = [os.path.join(data_dir, file) for file in os.listdir(data_dir) if os.path.isfile(os.path.join(data_dir, file))]  # .txt のみ抽出
filenames.sort()

save_to = "./data/embedding-vectors_exclusion/prott5/base"
os.makedirs(save_to, exist_ok=True)  # 保存先ディレクトリの作成

## 1.5 配列データのロード

In [ ]:
data = []

for filename in tqdm(filenames):
    with open(filename, "r", encoding="utf-8") as handle:
        for record in SeqIO.parse(filename, "fasta"):
            accession_number = record.id.split("|")[3] if len(record.id.split("|")[3]) > 0 else "Unknown"  # アクセッション番号を取得
            seq = record.seq

            # 有効なアミノ酸配列かを確認
            if check_valid_seq(seq):
                data.append((accession_number, seq))  # データに追加

  0%|          | 0/1 [00:00<?, ?it/s]

## 1.6 埋め込みベクトルの取得

In [ ]:
for i in tqdm(range(len(data))):
    aa = " ".join(list(data[i][1]))
    seq_len = len(data[i][1])

    # ids = tokenizer.batch_encode_plus(aa, add_special_tokens = True, pad_to_max_length = True)
    ids = tokenizer(aa, return_tensors='pt', add_special_tokens=True, padding="longest")
    input_ids = torch.tensor(ids['input_ids']).to(device)
    attention_mask = torch.tensor(ids["attention_mask"]).to(device)

    with torch.no_grad():
        embedding = model(input_ids = input_ids, attention_mask = attention_mask)
        embedding = embedding.last_hidden_state.cpu().numpy() #元々3次元リスト
        seq_emb = embedding[0][:seq_len]
        
    np.save(os.path.join(save_to, f"{i + 1}.npy"), seq_emb)

  0%|          | 0/7612 [00:00<?, ?it/s]

C:\Users\haivu\AppData\Local\Temp\ipykernel_54064\209969370.py:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_ids = torch.tensor(ids['input_ids']).to(device)
C:\Users\haivu\AppData\Local\Temp\ipykernel_54064\209969370.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  attention_mask = torch.tensor(ids["attention_mask"]).to(device)


## 1.7 取得した埋め込みベクトルの確認

In [ ]:
arr = np.load('./data/embedding-vectors_exclusion/prott5/base/1.npy')

print(arr)
print(arr.shape)

[[ 0.26400945 -0.19960661  0.19517106 ... -0.1998786   0.11153902
   0.25305852]
 [ 0.11789701 -0.26015744  0.07922492 ... -0.04559659  0.20102055
   0.22325413]
 [ 0.35468078 -0.14076096  0.10456698 ... -0.13755454  0.23028602
   0.2541323 ]
 ...
 [ 0.25936505 -0.01719121  0.17058846 ... -0.20705728  0.45314914
  -0.1370006 ]
 [ 0.00152625  0.02593009  0.0859358  ... -0.18980943  0.07346001
   0.2513557 ]
 [-0.15759227 -0.07089011  0.1853312  ... -0.22801755 -0.07385205
  -0.2137777 ]]
(418, 1024)
